In [ ]:
import altair as alt
import pandas as pd

# 1. Desactivar límite de filas
alt.data_transformers.disable_max_rows()

# 2. Cargar tu base de datos
df = pd.read_csv('datos_cuatro_comunas.csv')
df['Comuna'] = df['Comuna'].str.upper()
df_buses = df[df['Modo'].str.contains('BUS', case=False, na=False)]
df_agrupado = df_buses.groupby(['Media_hora', 'Comuna'])['Subidas_Promedio'].sum().reset_index()

# 3. CREAR EL "SENSOR" DEL RATÓN
# Esto le dice a Altair que detecte la hora más cercana a donde tienes el cursor
hover = alt.selection_point(
    fields=['Media_hora'],
    nearest=True,
    on='mouseover',
    empty='none'
)

# 4. CAPA 1: Las líneas normales de tu gráfico
lineas = alt.Chart(df_agrupado).mark_line(strokeWidth=3).encode(
    x=alt.X('Media_hora:O', 
            title='Horario de Madrugada y Mañana',
            axis=alt.Axis(labelAngle=-45, labelOverlap='parity', labelFont='Montserrat', titleFont='Montserrat', titleFontWeight='bold')),
    y=alt.Y('Subidas_Promedio:Q', 
            title='Cantidad Total de Pasajeros',
            axis=alt.Axis(format=',.0f', labelFont='Montserrat', titleFont='Montserrat', titleFontWeight='bold')),
    color=alt.Color('Comuna:N', legend=alt.Legend(labelFont='Montserrat', titleFont='Montserrat', symbolStrokeWidth=4))
)

# 5. CAPA 2: Los puntos que aparecen al pasar el ratón (con la información exacta)
puntos = lineas.mark_circle(size=70).encode(
    # Opacidad: se hacen visibles (1) solo cuando pasas el ratón, si no, invisibles (0)
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
    # El recuadro negro (Tooltip) que saltará con los datos exactos
    tooltip=[
        alt.Tooltip('Comuna:N', title='Comuna'),
        alt.Tooltip('Media_hora:O', title='Hora Exacta'),
        alt.Tooltip('Subidas_Promedio:Q', title='Total Pasajeros', format=',.0f')
    ]
).add_params(hover)

# 6. CAPA 3: La línea vertical (el "escáner") que sigue al ratón
regla = alt.Chart(df_agrupado).mark_rule(color='gray', strokeDash=[3, 3]).encode(
    x='Media_hora:O',
).transform_filter(hover)

# 7. JUNTAR TODAS LAS CAPAS Y GUARDAR COMO HTML PARA TU WEB
grafico_final = (lineas + puntos + regla).properties(
    title='Volumen de Pasajeros',
    width=800, 
    height=450
).configure_title(
    font='Arial Narrow', 
    fontSize=18,
    anchor='start',
    color='#FFCC00'
).configure_view(
    stroke=None
)

# Mostrarlo en Colab
grafico_final

# ¡ESTA ES LA MAGIA PARA TU PÁGINA WEB!
# Guarda el gráfico como un archivo web interactivo, no como una foto plana.
grafico_final.save('grafico_lineas.html')